In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle # <-- Added for saving the model
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, learning_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, balanced_accuracy_score
from xgboost import XGBClassifier
from scipy.stats import uniform, randint
import time

STEP 1: DATA LOADING AND PREPARATION (6 Features Only)

In [3]:
# Define your specific path
file_path = '/Users/elmoumouhi/code/Lisaht03/project_accidents/project_accidents/project_accidents/notebooks/Data/agg_df.csv'

try:
    print(f"Attempting to load data from: {file_path}")
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"Error: The file was not found at {file_path}")
    print("Please check the path and try again.")
    exit()

# 1. Sample the data
df_sampled = df.sample(n=20000, random_state=42).copy()

# Feature Selection
TARGET_COLUMN = 'injury_severity'
CORE_FEATURES = [
    'day_of_week', 'hour', 'department',
    'surface_condition',
    'road_category', 'speed_limit'
]

# Create the feature set X and target y
X = df_sampled[CORE_FEATURES]
y = df_sampled[TARGET_COLUMN]

# Encode the target
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)


Attempting to load data from: /Users/elmoumouhi/code/Lisaht03/project_accidents/project_accidents/project_accidents/notebooks/Data/agg_df.csv


# STEP 2: PREPROCESSING PIPELINE

In [4]:
numerical_features = make_column_selector(dtype_include=['int64', 'float64'])
categorical_features = make_column_selector(dtype_include=['object'])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)


# STEP 3: MODEL DEFINITION AND HYPERPARAMETER OPTIMIZATION

In [5]:

xgb = XGBClassifier(
    objective='multi:softmax',
    num_class=num_classes,
    eval_metric='merror',
    use_label_encoder=False,
    random_state=42
)

xgb_pipe = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', xgb)])

param_dist = {
    'classifier__n_estimators': randint(200, 800),
    'classifier__learning_rate': uniform(0.01, 0.15),
    'classifier__max_depth': randint(5, 10),
    'classifier__subsample': uniform(0.7, 0.2),
    'classifier__colsample_bytree': uniform(0.7, 0.2),
    'classifier__gamma': uniform(0, 0.5)
}

print("\nStarting Randomized Search CV...")
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    xgb_pipe, param_distributions=param_dist, n_iter=50,
    scoring='f1_macro', cv=cv_splitter, verbose=1, random_state=42, n_jobs=-1
)

start_time = time.time()
random_search.fit(X_train, y_train)
print(f"Randomized Search completed in {time.time() - start_time:.2f} seconds.")

best_model = random_search.best_estimator_


Starting Randomized Search CV...
Fitting 5 folds for each of 50 candidates, totalling 250 fits


/Users/elmoumouhi/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [12:17:51] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/elmoumouhi/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [12:17:51] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/elmoumouhi/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [12:17:51] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/elmoumouhi/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [12:17:51] WARNING: /Users/runner/work/xgboost

Randomized Search completed in 653.64 seconds.


# STEP 4: FINAL EVALUATION

In [6]:
print("\n--- Optimized XGBoost Model Performance (6 Features) ---")
y_pred = best_model.predict(X_test)

f1_test = f1_score(y_test, y_pred, average='macro')
bal_acc_test = balanced_accuracy_score(y_test, y_pred)

print(f"Final Test Set f1_macro Score: **{f1_test:.4f}**")
print(f"Final Test Set Balanced Accuracy: {bal_acc_test:.4f}")


--- Optimized XGBoost Model Performance (6 Features) ---
Final Test Set f1_macro Score: **0.4146**
Final Test Set Balanced Accuracy: 0.4098


# STEP 5: OVERFITTING CHECK (LEARNING CURVE)

In [7]:
print("\n--- Generating Learning Curve Plot ---")

train_sizes, train_scores, test_scores = learning_curve(
    estimator=best_model,
    X=X_train,
    y=y_train,
    cv=cv_splitter,
    scoring='f1_macro',
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    random_state=42
)

train_scores_mean = np.mean(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)

# Plotting the Learning Curve
plt.figure(figsize=(10, 6))
plt.title('Learning Curve (XGBoost with 6 Core Features)')
plt.xlabel('Training Examples')
plt.ylabel('F1-Macro Score')
plt.grid()

plt.plot(train_sizes, train_scores_mean, 'o-', color='r', label='Training Score')
plt.plot(train_sizes, test_scores_mean, 'o-', color='g', label='Cross-Validation Score')

plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std, alpha=0.1, color='r')
plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std, alpha=0.1, color='g')

plt.legend(loc='best')
plt.savefig('xgboost_learning_curve_6feat.png')
plt.close()

print("Learning curve saved as 'xgboost_learning_curve_6feat.png'.")


--- Generating Learning Curve Plot ---


/Users/elmoumouhi/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [12:28:34] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/elmoumouhi/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [12:28:34] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/elmoumouhi/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [12:28:34] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/elmoumouhi/.pyenv/versions/3.12.9/envs/lewagon/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [12:28:34] WARNING: /Users/runner/work/xgboost

Learning curve saved as 'xgboost_learning_curve_6feat.png'.


# STEP 6: SAVE THE TRAINED MODEL

In [8]:
model_filename = 'optimized_xgboost_6feat.pkl'
print(f"\n--- Saving Trained Model ---")

# Save the best_model (which is the full Pipeline including preprocessing)
with open(model_filename, 'wb') as f:
    pickle.dump(best_model, f)

print(f"Model saved successfully as '{model_filename}'.")


--- Saving Trained Model ---
Model saved successfully as 'optimized_xgboost_6feat.pkl'.
